# EnergyDB — Tree Diff Preview

Restructuring a portfolio is risky. `register_tree(..., dry_run=True)` returns a `TreeDiff` that lays out exactly what will change — inserts, renames, edits, deletes — as a tree, before anything touches the database. Same call, no surprises.

## 1. Setup

In [1]:
try:
    import urllib.request
    import google.colab  # noqa: F401

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rebase-energy/energydb/main/examples/colab_setup.py", "/tmp/colab_setup.py"
    )
    exec(open("/tmp/colab_setup.py").read())
except ImportError:
    pass

In [2]:
import energydb as edb

client = edb.Client()
client.delete()  # clean slate
client.create()

DatabaseError: Received ClickHouse exception, code: 194, server response: Code: 194. DB::Exception: default: Authentication failed: password is incorrect, or there is no user with such name.

If you use ClickHouse Cloud, the password can be reset at https://clickhouse.cloud/
on the settings page for the corresponding service.

If you have installed ClickHouse and forgot password you can reset it in the configuration file.
The password for default user is typically located at /etc/clickhouse-server/users.d/default-password.xml
and deleting this file will reset the password.
See also /etc/clickhouse-server/users.xml on the server where ClickHouse is installed.

. (REQUIRED_PASSWORD) (for url http://localhost:8123)

## 2. Persist a starting tree

A small offshore site with two turbines.

In [ ]:
portfolio = edb.Portfolio(
    name="my-portfolio",
    members=[
        edb.Site(
            name="Offshore-1",
            members=[
                edb.wind.WindTurbine(name="T01", capacity=3.5),
                edb.wind.WindTurbine(name="T02", capacity=3.5),
            ],
        ),
    ],
)
client.register_tree(portfolio)
print(client.get_tree("my-portfolio"))

## 3. Edit in memory, preview before committing

Round-trip the tree, mutate it freely — rename a site, tweak a property, drop a turbine, add a new one — then call `register_tree` with `dry_run=True`. The transaction is rolled back; you get a `TreeDiff` back instead of a uuid.

In [ ]:
tree = client.get_tree("my-portfolio")
tree.members[0].name = "Offshore-North"                                          # rename Site
tree.members[0].members[0].capacity = 4.0                                        # edit T01 capacity
del tree.members[0].members[1]                                                   # drop T02
tree.members[0].members.append(edb.wind.WindTurbine(name="T03", capacity=4.5))   # add T03

diff = client.register_tree(tree, mode="replace_subtree", allow_delete=True, dry_run=True)
print(f"has_changes: {diff.has_changes}\n")
diff.print()

## 4. Commit when satisfied

Drop the `dry_run` flag and the same call applies the changes.

In [ ]:
client.register_tree(tree, mode="replace_subtree", allow_delete=True)
print(client.get_tree("my-portfolio"))

## 5. Cleanup

In [ ]:
client.delete()